# R06 — User-study audit: what the logs actually show

**CPU only, ~1 min, no images and no network needed.** Re-derives every user-study number from the raw
session logs in `user_study/logs/` and reports what the logs support and what they do not. Addresses
reviewer comments on the user study (**R1** limitations of a 10-vs-12 design, **R2.7**: the study evaluated the
repair/fallback workflow rather than the parser, and *over-reliance was not measured*).

What this notebook establishes
1. **Session inventory** — which log files are complete sessions, which are stubs, which is a duplicate.
2. **Which parser actually ran at study time.** All interface sessions used `qwen/qwen-2.5-7b-instruct` via
   OpenRouter with the `llm_only_with_repair` parser and *never* triggered the keyword fallback. The paper's
   Table 8 "study-time" row (3B model, 87.5 % fallback, 5 % exact) describes a different parser.
3. **Parse correctness at study time**, scored without relying on the `used_fallback` flag (which cannot see
   semantic parse errors): numeric faithfulness of every parsed query to the participant's own text, and
   agreement with the task gold constraints (Table 9).
4. **Task 3 (relative constraint)** — whether any participant's *relative* request was actually represented.
5. **Table 10 recomputed.** In particular, "mean attempts" for condition B is checked against per-task counts.
6. **Descriptive statistics with uncertainty** (Wilson intervals, exact tests, effect sizes) for validity, time
   and questionnaire items, with the respondent accounting that the paper does not give.
7. **Over-reliance proxies** that can be computed from the existing logs.
8. **Assignment cohorts** — session start times by condition (the paper says "randomly assigned").

In [ ]:
# ============================================================
# C0) Common setup (identical in every review notebook)
# ============================================================
from pathlib import Path
import json, re, sys, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)
try:
    display
except NameError:
    display = print

SEED = 42
np.random.seed(SEED)

plt.rcParams.update({
    "font.family": "serif", "font.size": 8, "axes.labelsize": 8,
    "axes.titlesize": 8, "xtick.labelsize": 7, "ytick.labelsize": 7,
    "legend.fontsize": 7, "figure.dpi": 150, "savefig.dpi": 300,
    "axes.spines.top": False, "axes.spines.right": False,
})
SINGLE_COL = (3.35, 2.35)
WIDE_COL = (6.85, 2.65)

# ------------------------------------------------------------
# Robust repo-root detection (CWD, its parents, then rglob)
# ------------------------------------------------------------
CWD = Path.cwd().resolve()


def _is_repo(p: Path) -> bool:
    return (p / "mvtec" / "outputs_patchcore").is_dir() and (p / "proditec" / "outputs_patchcore").is_dir()


REPO = next((p for p in [CWD, *CWD.parents] if _is_repo(p)), None)
if REPO is None:
    for base in [CWD, CWD.parent]:
        for hit in base.rglob("outputs_patchcore"):
            if _is_repo(hit.parent.parent):
                REPO = hit.parent.parent
                break
        if REPO:
            break
if REPO is None:
    raise FileNotFoundError("Could not locate the repository root (needs mvtec/ and proditec/). "
                            "Launch Jupyter from the repo or from review/.")
REVIEW = REPO / "review"
OUT = REVIEW / "outputs"
OUT_TABLES, OUT_FIGS = OUT / "tables", OUT / "figures"
for d in (OUT_TABLES, OUT_FIGS):
    d.mkdir(parents=True, exist_ok=True)

DS = {
    "mvtec": dict(root=REPO / "mvtec", pre="mvtec",
                  data="surrogate_ready_dataset/mvtec_4class_patchcore_surrogate_dataset.csv"),
    "proditec": dict(root=REPO / "proditec", pre="proditec",
                     data="surrogate_ready_dataset/patchcore_surrogate_dataset_xgb.csv"),
}
for k, v in DS.items():
    r = v["root"]
    v["art"] = r / "surrogate_pkl_cfs_metadata"
    v["cfs"] = v["art"] / f"{k}_counterfactuals" / f"{k}_all_methods_cfs.csv"
    v["eval"] = v["art"] / f"{k}_counterfactuals_evaluation" / f"{k}_all_methods_cfs_patchcore_evaluation.csv"
    v["rev"] = v["art"] / f"{k}_revision_analysis"
    v["decomp"] = r / "revision_outputs" / "tables" / f"decomposition_per_candidate_{k}.csv"
    v["model"] = v["art"] / f"{k}_surrogate_catboost.cbm"
    v["meta"] = v["art"] / f"{k}_surrogate_metadata.json"
    v["dataset"] = r / v["data"]
    v["confirm"] = r / "revision_outputs" / "tables" / f"baseline_confirmation_{k}.csv"
    v["confirm_cands"] = r / "revision_outputs" / "tables" / f"baseline_confirm_candidates_{k}.csv"
print("REPO  :", REPO)
print("OUT   :", OUT)


def save_table(table: pd.DataFrame, stem: str, ff="%.4f"):
    p = OUT_TABLES / f"{stem}.csv"
    table.to_csv(p, index=False)
    try:
        table.to_latex(OUT_TABLES / f"{stem}.tex", index=False, escape=True, float_format=ff)
    except Exception as e:
        print("  (tex export skipped:", e, ")")
    print("Saved table:", p.relative_to(REPO))
    return p


def save_fig(fig, stem: str):
    for ext in ("pdf", "png"):
        fig.savefig(OUT_FIGS / f"{stem}.{ext}", bbox_inches="tight")
    plt.close(fig)
    print("Saved figure:", (OUT_FIGS / f"{stem}.pdf").relative_to(REPO))


def merge_numbers(section: str, values: dict):
    """Append to outputs/frozen_numbers.json (one section per notebook)."""
    p = OUT / "frozen_numbers.json"
    cur = json.loads(p.read_text()) if p.exists() else {}
    cur[section] = json.loads(json.dumps(values, default=float))
    p.write_text(json.dumps(cur, indent=2, sort_keys=True))
    print("Updated", p.relative_to(REPO), "->", section)

In [ ]:
# ============================================================
# R6.1) Load raw study logs
# ============================================================
import glob, hashlib, datetime as dt
from collections import Counter
from scipy import stats

US = REPO / "user_study"
LOGS_I, LOGS_N = US / "logs" / "interface", US / "logs" / "notebook"
assert LOGS_I.is_dir() and LOGS_N.is_dir(), "user_study/logs not found"

iface = {}      # participant -> session dict (interface, condition B)
iface_files = {}
for f in sorted(LOGS_I.glob("*_interface_session.json")):
    d = json.loads(f.read_text()); iface[d["participant_id"]] = d; iface_files[d["participant_id"]] = f.name
events_only = sorted({f.name.split("_")[0] for f in LOGS_I.glob("*_events.jsonl")} - set(iface))

nb = {}          # participant -> notebook session dict (condition A)
nb_hash = {}
for f in sorted(LOGS_N.glob("*_condition_a_answers.json")):
    d = json.loads(f.read_text()); nb[f.name.split("_")[0]] = d
    nb_hash[f.name.split("_")[0]] = hashlib.md5(f.read_bytes()).hexdigest()

export = pd.read_csv(US / "revision_outputs" / "task_level_answers_export.csv")
quest = pd.read_csv(next((US / "logs").glob("Post-Study Questionnaire*.csv")))
print(f"interface sessions: {len(iface)} (+{len(events_only)} events-only stubs: {events_only})")
print(f"notebook sessions : {len(nb)}")
print("export rows       :", len(export), "| questionnaire rows:", len(quest))

In [ ]:
# ============================================================
# R6.2) Session inventory: complete / stub / duplicate
# ============================================================
rows = []
for pid, d in iface.items():
    rows.append(dict(participant=pid, condition="B (conversational)", file=iface_files[pid],
                     start=d["session_started_at"], elapsed_min=d["total_elapsed_seconds"] / 60,
                     n_final_answers=len(d["final_answers"]), n_attempts=len(d["generated_attempts"]),
                     status="complete"))
for pid in events_only:
    ev = [json.loads(l) for l in (next(LOGS_I.glob(f"{pid}_*_events.jsonl"))).read_text().splitlines() if l.strip()]
    rows.append(dict(participant=pid, condition="B (conversational)", file=f"{pid}_events.jsonl only",
                     start=ev[0]["timestamp"], elapsed_min=ev[-1]["relative_seconds"] / 60, n_final_answers=0,
                     n_attempts=0, status=f"stub ({len(ev)} events, no task)"))
for pid, d in nb.items():
    dup = [q for q, h in nb_hash.items() if h == nb_hash[pid] and q != pid]
    t = d["timing"]
    _fn = next(LOGS_N.glob(f"{pid}_*_condition_a_answers.json")).name
    _m = re.search(r"_(\d{8})_(\d{6})_", _fn)
    rows.append(dict(participant=pid, condition="A (notebook/API)", file=f"{pid}_condition_a_answers.json",
                     start=dt.datetime.strptime(_m.group(1) + _m.group(2), "%Y%m%d%H%M%S").isoformat(), elapsed_min=t["notebook_total_seconds_at_save"] / 60, n_final_answers=len(d["answers"]),
                     n_attempts=len(d["attempt_log"]), status="duplicate-of-" + dup[0] if dup else "complete"))
inv = pd.DataFrame(rows)
inv["start"] = pd.to_datetime(inv["start"].astype(str).str.slice(0, 19))
inv = inv.sort_values(["condition", "start"]).reset_index(drop=True)
display(inv.round(2))
save_table(inv, "R06_session_inventory")

In [ ]:
# ============================================================
# R6.3) Which parser ran at study time?
# ============================================================
att = []
for pid, d in iface.items():
    for a in d["generated_attempts"]:
        q = a["query_template"]; m = q.get("_meta", {})
        att.append(dict(participant=pid, attempt=a["attempt_number"], task=a["task_name"], t=a["relative_seconds"],
                        text=" ".join(a["user_text"].split()), n_options=a["n_options"],
                        model=q.get("llm_model"), provider=q.get("llm_provider"), parser=m.get("parser"),
                        used_fallback=bool(m.get("used_fallback")), repair_attempts=m.get("attempts"),
                        app_version=d["app_version"], query=q, explanation=a.get("explanation", "")))
A = pd.DataFrame(att)
parser_tbl = (A.groupby(["provider", "model", "parser", "app_version"], dropna=False)
              .agg(attempts=("attempt", "size"), sessions=("participant", "nunique"),
                   fallback_rate_pct=("used_fallback", lambda s: 100 * s.mean()),
                   mean_repair_attempts=("repair_attempts", "mean")).reset_index())
display(parser_tbl)
save_table(parser_tbl, "R06_study_time_parser")

# The paper's Table 8 study-time row (3B model) for comparison
paper_t8 = pd.DataFrame([dict(source="paper Table 8, 'study-time' row", model="3B (local)", fallback_pct=87.5, exact_pct=5.0),
                         dict(source="study logs (this notebook)", model=", ".join(A.model.unique()),
                              fallback_pct=100 * A.used_fallback.mean(), exact_pct=np.nan)])
display(paper_t8)
save_table(paper_t8, "R06_parser_row_paper_vs_logs")
OUTN = dict(n_attempts=int(len(A)), n_fallback=int(A.used_fallback.sum()), models=sorted(A.model.unique().tolist()),
            parsers=sorted(A.parser.unique().tolist()))
print(OUTN)

In [ ]:
# ============================================================
# R6.4) Parse correctness at study time, independent of the `used_fallback` flag
# ============================================================
# (a) numeric faithfulness : every number the parser emitted must occur in the participant's text, and every
#     number in the text should be represented in the parse
# (b) task-gold alignment  : does the parsed query equal the constraints of the task (Table 9)?
TOL = 0.0126            # budget/fraction values are snapped to a 0.005/0.01 grid by the app
GOLD = {                # (objective, budget upper, train lower, train upper, corruption)
    "task_1": dict(obj=("band", 0.80, 0.92), b_hi=0.40, t_lo=0.40, t_hi=0.70, corr=None),
    "task_2": dict(obj=("band", 0.78, 0.91), b_hi=0.30, t_lo=0.20, t_hi=0.60, corr={"mild", "none"}),
    # Task 3 is *relative*; an absolute query is equivalent to it iff F1 >= base (0.84984) and budget <= 0.356
    "task_3": dict(obj=("min", 0.84984, None), b_hi=0.35625, t_lo=None, t_hi=None, corr=None),
}
DEFAULT_LO = {"b": 0.005, "t": 0.2}


def text_numbers(txt):
    t = txt.replace(",", ".")
    nums = []
    for m in re.finditer(r"(\d*\.?\d+)\s*(%?)", t):
        v = float(m.group(1)) if m.group(1) != "." else np.nan
        nums.append(v / 100 if m.group(2) else v)
        if not m.group(2):
            nums.append(v)
    return [x for x in nums if np.isfinite(x)]


def parsed_numbers(q):
    o, s = q["objective"], q["soft_constraints"]
    out = []
    for k in ("lower", "upper", "value", "delta"):
        if k in o: out.append(o[k])
    for key, dflt in (("params_soft_review_budget", DEFAULT_LO["b"]), ("params_soft_train_fraction", DEFAULT_LO["t"])):
        c = s.get(key, {})
        if c.get("mode") in ("range", "fixed", "at_most", "at_least"):
            for k in ("lower", "upper", "value"):
                if k in c and not (k == "lower" and abs(c[k] - dflt) < 1e-9):
                    out.append(c[k])
    return out


def covered(x, pool):
    return any(abs(x - y) <= TOL for y in pool)


def faithfulness(text, q):
    tn, pn = text_numbers(text), parsed_numbers(q)
    implicit = [DEFAULT_LO["b"], DEFAULT_LO["t"]]       # app-side default lower bounds count as 'represented'
    spurious = [round(p, 3) for p in pn if not covered(p, tn)]
    dropped = [x for x in sorted(set(round(v, 3) for v in tn))
               if x not in (0.0, 1.0) and not covered(x, pn + implicit)]
    return spurious, dropped


REL_RE = re.compile(r"%|base configuration|reduc\w+ |reduced|from 0[.,]375|lower than 0[.,]375|at least 5", re.I)


def gold_match(task, q):
    g = GOLD[task]; o, s = q["objective"], q["soft_constraints"]
    ok = {}
    if g["obj"][0] == "band":
        ok["objective"] = o["type"] == "target_band" and abs(o["lower"] - g["obj"][1]) < 0.011 and abs(o["upper"] - g["obj"][2]) < 0.011
    else:                     # task 3: absolute equivalent of "F1 >= base"
        ok["objective"] = (o["type"] == "target_min" and abs(o["value"] - g["obj"][1]) < 0.011) or \
                          (o["type"] == "target_band" and abs(o["lower"] - g["obj"][1]) < 0.011)
    b = s.get("params_soft_review_budget", {})
    ok["budget"] = b.get("mode") == "range" and b["upper"] <= g["b_hi"] + (TOL if g["b_hi"] > 0.36 else 0.0) + 1e-9 \
        if g["b_hi"] == 0.35625 else (b.get("mode") == "range" and abs(b["upper"] - g["b_hi"]) <= TOL)
    if g["t_lo"] is not None:
        t = s.get("params_soft_train_fraction", {})
        ok["train"] = t.get("mode") == "range" and abs(t["lower"] - g["t_lo"]) <= TOL and abs(t["upper"] - g["t_hi"]) <= TOL
    c = s.get("params_soft_corruption_level", {})
    got = None if c.get("mode") == "free" or not c else set(c.get("allowed", [c.get("value")]))
    ok["corruption"] = got == g["corr"] or (g["corr"] is None and got is None)
    return ok


recs = []
for _, r in A.iterrows():
    sp, dr = faithfulness(r.text, r.query)
    gm = gold_match(r.task, r.query)
    recs.append(dict(spurious_values=sp, dropped_values=dr, numerically_faithful=len(sp) == 0,
                     fully_captured=(len(sp) == 0 and len(dr) == 0), relative_phrasing=bool(REL_RE.search(r.text)),
                     task_gold_exact=all(gm.values()), **{f"gold_{k}": v for k, v in gm.items()},
                     obj_type=r.query["objective"]["type"]))
A = pd.concat([A.drop(columns=["spurious_values", "dropped_values"], errors="ignore"), pd.DataFrame(recs)], axis=1)
A["zero_options"] = A.n_options == 0


def _sentinel(q):
    b = q["soft_constraints"].get("params_soft_review_budget", {})
    return b.get("mode") == "range" and b.get("lower") == 0.5 and b.get("upper") == 0.5


A["sentinel_budget_0p5"] = A["query"].map(_sentinel)
A["semantic_error_not_fallback"] = (~A.numerically_faithful) & (~A.used_fallback)

summ = (A.groupby("task").agg(attempts=("attempt", "size"), fallback=("used_fallback", "sum"),
                              numerically_unfaithful=("numerically_faithful", lambda s: int((~s).sum())),
                              not_fully_captured=("fully_captured", lambda s: int((~s).sum())),
                              zero_options=("zero_options", "sum"), relative_phrased=("relative_phrasing", "sum"),
                              sentinel_budget_0p5=("sentinel_budget_0p5", "sum"),
                              task_gold_exact=("task_gold_exact", "sum")).reset_index())
tot = summ.drop(columns="task").sum().to_frame().T; tot.insert(0, "task", "all")
summ = pd.concat([summ, tot], ignore_index=True)
display(summ)
save_table(summ, "R06_parse_audit_summary")
cols = ["participant", "attempt", "task", "text", "obj_type", "n_options", "used_fallback", "numerically_faithful",
        "spurious_values", "dropped_values", "relative_phrasing", "task_gold_exact"]
save_table(A[cols], "R06_parse_audit_per_attempt")
print("\nAttempts where the parser silently produced values absent from the participant's text (not a fallback):")
display(A.loc[A.semantic_error_not_fallback, ["participant", "attempt", "task", "text", "spurious_values"]])

In [ ]:
# ============================================================
# R6.5) Task 3: was the *relative* request ever represented?
# ============================================================
t3 = A[A.task == "task_3"].copy()
by_p = []
for pid, g in t3.groupby("participant"):
    g = g.sort_values("attempt")
    fa = iface[pid]["final_answers"]
    fin = next((x for x in fa if x["task_name"] == "task_3"), None)
    fq = fin["current_query_template"] if fin else None
    by_p.append(dict(participant=pid, task3_attempts=len(g), relative_phrased_attempts=int(g.relative_phrasing.sum()),
                     relative_attempts_with_0_options=int((g.relative_phrasing & g.zero_options).sum()),
                     final_query_type=(fq["objective"]["type"] if fq else None),
                     final_query_budget_upper=(fq["soft_constraints"]["params_soft_review_budget"].get("upper") if fq else None),
                     final_query_absolute_equivalent=bool(fq and gold_match("task_3", fq)["objective"] and gold_match("task_3", fq)["budget"]),
                     final_selection_valid=(fin["analysis_fields"]["valid_selection"] if fin else None),
                     system_said_all_constraints_met=bool(fin and fin["current_explanation"].lower().startswith(("all options meet", "all shown options")))))
T3 = pd.DataFrame(by_p)
display(T3)
save_table(T3, "R06_task3_relative_query_handling")
n_rel_att = int(t3.relative_phrasing.sum())
n_rel_rep = int((t3.relative_phrasing & ~t3.zero_options & t3.fully_captured).sum())
print(f"\nTask-3 attempts phrased relatively: {n_rel_att}; of these, {int((t3.relative_phrasing & t3.zero_options).sum())} returned "
      f"0 options; {n_rel_rep} produced a query whose numbers all match the text (absolute rewrites such as 'from 0.375 to 0.37'), "
      f"none produced a representable *relative* budget constraint (the schema has no such field; the app falls back to the sentinel budget range [0.5, 0.5]: "
      f"{int(t3.sentinel_budget_0p5.sum())}/{len(t3)} Task-3 attempts).")
print("Participants whose final accepted query was an *absolute* rewrite:", int(T3.final_query_absolute_equivalent.sum()), "/", len(T3))
print("Final queries with budget upper above the 5 % target (0.356):",
      int((T3.final_query_budget_upper.fillna(0) > 0.3563).sum()))

In [ ]:
# ============================================================
# R6.6) Table 10 recomputed, and the 'mean attempts' definition
# ============================================================
ex = export.copy()
ex["cond"] = ex.condition.map({"condition_a": "A", "condition_llm_interface": "B"})
# --- per-task attempts -------------------------------------------------------------------------------------
att_rows = []
for pid, d in nb.items():
    if nb_hash[pid] in [nb_hash[q] for q in nb if q < pid]:    # drop the byte-identical duplicate session
        continue
    for k in (1, 2, 3):
        att_rows.append(dict(participant=pid, cond="A", task=f"task_{k}", attempts_per_task=d["timing"][f"task_{k}_attempt_count"]))
for pid, d in iface.items():
    cnt = Counter(a["task_name"] for a in d["generated_attempts"])
    for fa in d["final_answers"]:
        att_rows.append(dict(participant=pid, cond="B", task=fa["task_name"], attempts_per_task=cnt[fa["task_name"]],
                             attempts_cumulative_in_session=fa["attempt_count_at_submission"], _t=fa["relative_seconds"]))
AT = pd.DataFrame(att_rows)
# a final answer can be re-submitted (P-3FA4E7 task 2): keep the last submission per participant x task
AT = AT.sort_values("_t").drop_duplicates(["participant", "cond", "task"], keep="last").drop(columns="_t")
m = ex.merge(AT, left_on=["participant_id", "cond", "task_name"], right_on=["participant", "cond", "task"], how="left")
m["f1_valid"] = np.where(m.valid_selection, m.selected_quality, np.nan)
t10 = (m.groupby(["cond", "task_name"]).agg(n=("participant_id", "size"), valid_pct=("valid_selection", lambda s: 100 * s.mean()),
                                            attempts_per_task=("attempts_per_task", "mean"),
                                            attempts_cumulative=("attempts_cumulative_in_session", "mean"),
                                            f1_all=("selected_quality", "mean"), f1_valid=("f1_valid", "mean"),
                                            budget=("selected_review_budget", "mean")).reset_index())
paper_att = {("A", "task_1"): 1.6, ("A", "task_2"): 1.2, ("A", "task_3"): 1.2, ("B", "task_1"): 1.583,
             ("B", "task_2"): 3.5, ("B", "task_3"): 6.273}
t10["paper_mean_attempts"] = [paper_att[(c, t)] for c, t in zip(t10.cond, t10.task_name)]
display(t10.round(3))
save_table(t10, "R06_table10_recomputed")
print("\nPaper's condition-B 'mean attempts' (3.5, 6.27) equals the *cumulative* session attempt count at submission, "
      "not the per-task count:",
      np.allclose(t10.loc[t10.cond == "B", "attempts_cumulative"].values, t10.loc[t10.cond == "B", "paper_mean_attempts"].values, atol=0.01))

In [ ]:
# ============================================================
# R6.7) Validity, time and Task-3 outcome with uncertainty (descriptive; n = 10 vs 12)
# ============================================================
def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p = k / n; den = 1 + z * z / n
    c = (p + z * z / (2 * n)) / den; h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return c - h, c + h


def cliffs_delta(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    return float(np.mean([int(x > y) - int(x < y) for x in a for y in b]))


def boot_ci(fn, a, b, n=5000, seed=SEED):
    rng = np.random.default_rng(seed); a, b = np.asarray(a, float), np.asarray(b, float)
    v = [fn(rng.choice(a, len(a)), rng.choice(b, len(b))) for _ in range(n)]
    return float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))


rows = []
for task, g in ex.groupby("task_name"):
    a, b = g[g.cond == "A"].valid_selection.astype(int), g[g.cond == "B"].valid_selection.astype(int)
    la, ha = wilson(a.sum(), len(a)); lb, hb = wilson(b.sum(), len(b))
    table = [[a.sum(), len(a) - a.sum()], [b.sum(), len(b) - b.sum()]]
    rows.append(dict(task=task, nA=len(a), validA=int(a.sum()), pctA=100 * a.mean(), ciA=f"[{100*la:.0f}, {100*ha:.0f}]",
                     nB=len(b), validB=int(b.sum()), pctB=100 * b.mean(), ciB=f"[{100*lb:.0f}, {100*hb:.0f}]",
                     fisher_p=stats.fisher_exact(table)[1]))
V = pd.DataFrame(rows)
display(V.round(3)); save_table(V, "R06_validity_wilson_fisher")

# pooled over tasks (cluster-aware: participant is the unit)
pa = ex[ex.cond == "A"].groupby("participant_id").valid_selection.mean()
pb = ex[ex.cond == "B"].groupby("participant_id").valid_selection.mean()
print(f"\nParticipant-level all-tasks-valid rate: A {np.mean(pa==1):.2f} (n={len(pa)}), B {np.mean(pb==1):.2f} (n={len(pb)}); "
      f"Fisher p = {stats.fisher_exact([[int((pa==1).sum()), int((pa<1).sum())],[int((pb==1).sum()), int((pb<1).sum())]])[1]:.3f}")

# completion time
tm = []
for pid, d in nb.items():
    if nb_hash[pid] in [nb_hash[q] for q in nb if q < pid]: continue
    tm.append(dict(participant=pid, cond="A", minutes=d["timing"]["notebook_total_seconds_at_save"] / 60))
for pid, d in iface.items():
    tm.append(dict(participant=pid, cond="B", minutes=d["total_elapsed_seconds"] / 60))
TM = pd.DataFrame(tm)
ta, tb = TM[TM.cond == "A"].minutes.values, TM[TM.cond == "B"].minutes.values
u = stats.mannwhitneyu(ta, tb, alternative="two-sided")
tm_tbl = pd.DataFrame([dict(cond=c, n=len(x), mean=x.mean(), median=np.median(x), sd=x.std(ddof=1), min=x.min(), max=x.max())
                       for c, x in (("A", ta), ("B", tb))])
tm_tbl["mannwhitney_p"] = u.pvalue; tm_tbl["cliffs_delta_B_minus_A"] = cliffs_delta(tb, ta)
lo, hi = boot_ci(lambda x, y: np.median(y) - np.median(x), ta, tb)
tm_tbl["median_diff_B_minus_A_ci95"] = f"[{lo:.1f}, {hi:.1f}]"
tm_tbl["paper_mean_min"] = [23.64, 23.31]; tm_tbl["paper_sd_min"] = [14.69, 8.90]
display(tm_tbl.round(3)); save_table(tm_tbl, "R06_completion_time")

# Task 3 outcome
t3o = ex[ex.task_name == "task_3"].copy()
t3o["budget_reduction_pct"] = 100 * (0.375 - t3o.selected_review_budget) / 0.375
t3o["dq"] = t3o.selected_quality - 0.84984
t3s = t3o.groupby("cond").agg(n=("participant_id", "size"), valid_pct=("valid_selection", lambda s: 100 * s.mean()),
                              mean_budget_reduction_pct=("budget_reduction_pct", "mean"),
                              median_budget_reduction_pct=("budget_reduction_pct", "median"),
                              mean_dq=("dq", "mean"), median_dq=("dq", "median")).reset_index()
display(t3s.round(3)); save_table(t3s, "R06_task3_outcome")

In [ ]:
# ============================================================
# R6.8) Questionnaire: respondent accounting, item-level tests, effect sizes
# ============================================================
LIK = {"Strongly disagree": 1, "Disagree": 2, "Neutral": 3, "Agree": 4, "Strongly agree": 5}
q = quest.copy(); q.columns = ["ts", "pid"] + ["identify", "check", "tradeoffs", "compare", "use"]
q["cond"] = q.pid.map(lambda p: "B" if p in iface else ("A" if p in nb else "unmatched"))
acct = q.groupby("cond").agg(rows=("pid", "size"), unique=("pid", "nunique")).reset_index()
print("Questionnaire rows by matched condition:"); display(acct)
print("Unmatched ids:", q.loc[q.cond == "unmatched", "pid"].tolist())
print("Duplicated ids:", q.pid[q.pid.duplicated(keep=False)].tolist(), "-> kept first submission")
print("Sessions without any questionnaire row: A:", sorted(set(nb) - set(q.pid) - {"P-D2166E1"}), "| B:", sorted(set(iface) - set(q.pid)))
qq = q[q.cond.isin(["A", "B"])].drop_duplicates("pid", keep="first").copy()
items = ["identify", "check", "tradeoffs", "compare", "use"]
for c in items: qq[c] = qq[c].map(LIK)
rows = []
for c in items:
    a, b = qq[qq.cond == "A"][c].values, qq[qq.cond == "B"][c].values
    pu = stats.mannwhitneyu(a, b, alternative="two-sided")
    lo, hi = boot_ci(lambda x, y: y.mean() - x.mean(), a, b)
    rows.append(dict(item=c, nA=len(a), nB=len(b), meanA=a.mean(), meanB=b.mean(), diff=b.mean() - a.mean(),
                     diff_ci95=f"[{lo:.2f}, {hi:.2f}]", cliffs_delta=cliffs_delta(b, a), mannwhitney_p=pu.pvalue))
Q = pd.DataFrame(rows)
_o = np.argsort(Q.mannwhitney_p.values); _m = len(Q)
_adj = np.maximum.accumulate([min(1.0, (_m - i) * Q.mannwhitney_p.values[k]) for i, k in enumerate(_o)])
Q["p_holm"] = np.empty(_m); Q.loc[_o, "p_holm"] = _adj
display(Q.round(3)); save_table(Q, "R06_questionnaire_tests")
print("\nNote: with 9 vs 11 respondents and five items none of the differences survives Holm correction; the "
      "'compare options' difference also confounds the interaction mode with the presentation format "
      "(ranked table + textual trade-off summary vs. notebook candidate list).")

In [ ]:
# ============================================================
# R6.9) Over-reliance proxies (R2.7): what the existing logs can and cannot say
# ============================================================
REC = re.compile(r"balanced[^.\n]*?option\s*([ABC])", re.I)
ob = []
for pid, d in iface.items():
    last = {}
    for fa in sorted(d["final_answers"], key=lambda z: z["relative_seconds"]):
        last[fa["task_name"]] = fa          # keep the last submission per task (35 answers in total)
    for fa in last.values():
        af = fa["analysis_fields"]; ex_txt = fa["current_explanation"]
        rec = REC.search(ex_txt)
        gm = gold_match(fa["task_name"], fa["current_query_template"])
        ob.append(dict(participant=pid, task=fa["task_name"], selected=af["selected_option_name"],
                       recommended_balanced=(f"Option {rec.group(1).upper()}" if rec else None),
                       followed_recommendation=(rec is not None and af["selected_option_name"] == f"Option {rec.group(1).upper()}"),
                       selected_is_top_quality=af["quality_rank_within_displayed_options"] == 1,
                       selected_is_lowest_budget=af["review_budget_rank_within_displayed_options"] == 1,
                       query_matches_task_gold=all(gm.values()),
                       system_claimed_all_met=ex_txt.lower().startswith(("all options meet", "all shown options")),
                       valid=bool(af["valid_selection"]),
                       invalid_but_system_claimed_met=(not af["valid_selection"]) and ex_txt.lower().startswith(("all options meet", "all shown options")),
                       n_displayed=af["displayed_option_count"], source=af["selected_source"]))
OB = pd.DataFrame(ob)
# condition A analogue: rank of the selected run inside the displayed list
oa = []
for pid, d in nb.items():
    if nb_hash[pid] in [nb_hash[q] for q in nb if q < pid]: continue
    for a_ in d["answers"]:
        n_disp = sum(1 for z in d["displayed_order"] if z["task_name"] == a_["task_name"])
        oa.append(dict(participant=pid, task=a_["task_name"], q_rank=a_["quality_rank_within_displayed_options"],
                       b_rank=a_["review_budget_rank_within_displayed_options"], n_disp=n_disp,
                       valid=bool(a_["valid_selection"])))
OA = pd.DataFrame(oa)
display(OB[["participant", "task", "selected", "recommended_balanced", "followed_recommendation", "selected_is_top_quality",
            "query_matches_task_gold", "system_claimed_all_met", "valid"]])
summ = pd.DataFrame([
    dict(metric="B: selections following the interface's 'balanced' recommendation", k=int(OB.followed_recommendation.sum()), n=int(OB.recommended_balanced.notna().sum())),
    dict(metric="B: selected option is the top-quality displayed option", k=int(OB.selected_is_top_quality.sum()), n=len(OB)),
    dict(metric="B: selected option is the lowest-budget displayed option", k=int(OB.selected_is_lowest_budget.sum()), n=len(OB)),
    dict(metric="B: final-answer query equals the task gold constraints", k=int(OB.query_matches_task_gold.sum()), n=len(OB)),
    dict(metric="B: invalid selections although the interface said all options met the constraints", k=int(OB.invalid_but_system_claimed_met.sum()), n=int((~OB.valid).sum())),
    dict(metric="B: all selections come from retrieval (dataset_filter), none from generation", k=int((OB.source == "dataset_filter").sum()), n=len(OB)),
    dict(metric=f"A: median quality rank of the selected run among a list of ~{int(OA.n_disp.median())} (rank 1 = best)", k=int(OA.q_rank.median()), n=int(OA.n_disp.median())),
])
summ["rate_pct"] = 100 * summ.k / summ.n
summ.loc[summ.metric.str.startswith("A:"), "rate_pct"] = np.nan
display(summ.round(1)); save_table(summ, "R06_overreliance_proxies")
print("\nCaveat: 'followed the recommendation' is not evidence of over-reliance unless the recommendation was wrong. "
      "A direct measure (planted wrong recommendations / persuasive vs neutral labelling) was not part of this study "
      "and cannot be reconstructed from the logs; it is reported as a limitation and as future work.")

In [ ]:
# ============================================================
# R6.10) Cohort structure: was assignment individual or per laboratory block?
# ============================================================
c = inv[inv.status.isin(["complete"]) | inv.status.str.startswith("duplicate")].copy()
c["date"] = c.start.dt.date; c["hhmm"] = c.start.dt.strftime("%H:%M")
cohort = (c.groupby(["condition", "date"]).agg(sessions=("participant", "size"), first_start=("start", "min"), last_start=("start", "max")).reset_index())
cohort["window_min"] = (cohort.last_start - cohort.first_start).dt.total_seconds() / 60
display(cohort)
save_table(cohort, "R06_cohort_structure")
overlap = not (c[c.condition.str.startswith("A")].start.min() <= c[c.condition.str.startswith("B")].start.max() and
               c[c.condition.str.startswith("B")].start.min() <= c[c.condition.str.startswith("A")].start.max())
print("Condition windows overlap in time:", not overlap, "| All condition-B sessions fall in one window of",
      round(float(cohort[cohort.condition.str.startswith('B')].window_min.max()), 1), "min")
print("=> Conditions look like two laboratory blocks (B first, then A). Randomisation at the individual level cannot be "
      "verified from the logs; the manuscript should say 'allocated by session block' unless the allocation record exists.")

In [ ]:
# ============================================================
# R6.11) Figures
# ============================================================
fig, ax = plt.subplots(1, 2, figsize=WIDE_COL)
w = 0.35; xs = np.arange(3)
for j, (cond, col) in enumerate((("A", "#7c8794"), ("B", "#3b82d4"))):
    k = [int(V.loc[V.task == f"task_{i}", f"valid{cond}"].iloc[0]) for i in (1, 2, 3)]
    n = [int(V.loc[V.task == f"task_{i}", f"n{cond}"].iloc[0]) for i in (1, 2, 3)]
    p = np.array(k) / np.array(n); ci = np.array([wilson(a, b) for a, b in zip(k, n)])
    ax[0].bar(xs + (j - 0.5) * w, 100 * p, w, color=col, label=f"{cond} ({'notebook' if cond=='A' else 'conversational'})")
    ax[0].errorbar(xs + (j - 0.5) * w, 100 * p, yerr=[100 * (p - ci[:, 0]), 100 * (ci[:, 1] - p)], fmt="none", ecolor="k", lw=0.8, capsize=2)
ax[0].set_xticks(xs); ax[0].set_xticklabels(["Task 1", "Task 2", "Task 3"]); ax[0].set_ylabel("Valid selections (%)")
ax[0].set_ylim(0, 125); ax[0].set_yticks([0, 20, 40, 60, 80, 100]); ax[0].legend(frameon=False, loc="upper center", ncol=2); ax[0].set_title("Validity with 95 % Wilson intervals")
for j, (cond, col) in enumerate((("A", "#7c8794"), ("B", "#3b82d4"))):
    d_ = Q[["item", "meanA", "meanB"]]
ax[1].barh(np.arange(len(Q)) + 0.18, Q.meanA, 0.36, color="#7c8794", label="A")
ax[1].barh(np.arange(len(Q)) - 0.18, Q.meanB, 0.36, color="#3b82d4", label="B")
ax[1].set_yticks(np.arange(len(Q))); ax[1].set_yticklabels(["identify F1", "check constr.", "trade-offs", "compare opt.", "use tool"])
ax[1].invert_yaxis(); ax[1].set_xlim(1, 5); ax[1].set_xlabel("Mean Likert (1–5)"); ax[1].set_title(f"Ease items (n = {int(Q.nA[0])} vs {int(Q.nB[0])})")
fig.tight_layout(); save_fig(fig, "R06_validity_and_questionnaire")

fig, ax = plt.subplots(figsize=SINGLE_COL)
bb = t10[t10.cond == "B"]
ax.bar(np.arange(3) - 0.18, bb.attempts_per_task, 0.36, color="#3b82d4", label="per task (logs)")
ax.bar(np.arange(3) + 0.18, bb.paper_mean_attempts, 0.36, color="#c9d3df", label="as printed in Table 10")
ax.set_xticks(range(3)); ax.set_xticklabels(["Task 1", "Task 2", "Task 3"]); ax.set_ylabel("Mean attempts (condition B)")
ax.legend(frameon=False); fig.tight_layout(); save_fig(fig, "R06_attempts_definition")

In [ ]:
# ============================================================
# R6.12) Freeze numbers
# ============================================================
merge_numbers("R06_user_study_audit", dict(
    interface_sessions_complete=len(iface), events_only_stubs=events_only, notebook_sessions=len(nb),
    notebook_duplicate=[p for p in nb if nb_hash[p] in [nb_hash[q] for q in nb if q < p]],
    attempts_total=OUTN["n_attempts"], fallback_attempts=OUTN["n_fallback"], models=OUTN["models"], parsers=OUTN["parsers"],
    semantic_parse_errors_not_fallback=int(A.semantic_error_not_fallback.sum()),
    attempts_numerically_unfaithful=int((~A.numerically_faithful).sum()),
    task3_relative_phrased_attempts=n_rel_att, task3_relative_fully_captured=n_rel_rep,
    task3_final_absolute_rewrites=int(T3.final_query_absolute_equivalent.sum()), task3_participants=len(T3),
    table10=t10.round(4).to_dict(orient="records"), questionnaire=Q.round(4).to_dict(orient="records"),
    followed_recommendation=int(OB.followed_recommendation.sum()), recommendation_n=int(OB.recommended_balanced.notna().sum()),
))